# LangChain Basics

LangChain wraps everything you've done manually so far
into clean, reusable building blocks.

Raw API way (what you did before):
  client.chat.completions.create(model=..., messages=[...])

LangChain way:
  llm.invoke("your message")

Same result — cleaner code — more powerful when chaining.

# Installation

In [1]:
from urllib import response

from langchain_core import messages
!pip install langchain langchain-groq python-dotenv

  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.18-py3-none-any.whl.metadata (2.4 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached uuid_utils-0.17.0-cp312-cp312-win_amd64.whl.metadata (6.5 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached xxhash-3.8.1-cp312-cp312-win_amd64.whl.metadata (15 kB)
  Using cached ormsgpack-1.12.2-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
  Using cached orjson-3.11.9-cp312-cp312-win_amd64.whl.metadata (43 kB)
  Using cached websockets-15.0.1-cp312-cp312-win_amd64.whl.metadata (7.0 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Setup

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
    api_key=os.getenv("GROQ_API_KEY")
)

print("Langcahin and Groq ready!")

Langcahin and Groq ready!


# Basic invocation

In [3]:
response = llm.invoke("What is Langchain in one sentence?")
print(response.content)
print(type(response))

LangChain is an open‑source framework that streamlines the construction of AI applications by providing modular, reusable components for integrating, orchestrating, and extending large language models.
<class 'langchain_core.messages.ai.AIMessage'>


# With messages

In [4]:
messages = [
    SystemMessage(content="You are a concise Python tutor."),
    HumanMessage(content="What is a decorator in Python?")
]

response = llm.invoke(messages)
print(response.content)

A **decorator** is a function that takes another function (or class) and returns a new one, usually adding or modifying its behavior.  
In practice it lets you wrap code around a function without changing the function itself.

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("Before")
        result = func(*args, **kwargs)
        print("After")
        return result
    return wrapper

@my_decorator          # same as: my_func = my_decorator(my_func)
def my_func(x):
    return x * 2

print(my_func(5))  # prints Before, After, then 10
```

Key points:
- `@decorator_name` is syntactic sugar for `func = decorator_name(func)`.
- Decorators can add logging, timing, authentication, memoization, etc., keeping the original function clean.


## Message Types

HumanMessage  → role: "user"

SystemMessage → role: "system"

AIMessage     → role: "assistant"

These are the same roles as raw API — just wrapped in classes.
LangChain uses objects instead of dicts — cleaner and type-safe.

# Prompt Templates

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {subject}. Be concise and clear."),
    ("human", "{question}")
])

filled_prompt = prompt.invoke({
    "subject": "Python",
    "question": "What is a generator?"
})

print("Filled prompt messages:")
for msg in filled_prompt.messages:
    print(f"{msg.type}: {msg.content}")

Filled prompt messages:
system: You are an expert in Python. Be concise and clear.
human: What is a generator?


# Template with LLM

In [6]:
response = llm.invoke(filled_prompt)
print("LLM Response:")
print(response.content)

LLM Response:
A **generator** is a special kind of iterator in Python that produces values on the fly instead of computing them all at once.

* **Definition** – A generator is created by a function that contains one or more `yield` statements, or by a generator expression (`(x for x in iterable)`).

* **How it works** – Each time the generator’s `__next__()` (or `next()`) is called, the function runs until it hits the next `yield`. The yielded value is returned, and the function’s state is saved. When the function finishes, `StopIteration` is raised.

* **Benefits**  
  * **Memory‑efficient** – only one value is stored at a time.  
  * **Lazy evaluation** – values are produced only when needed.  
  * **Composable** – you can chain generators for clean, readable pipelines.

```python
def squares(n):
    for i in range(n):
        yield i * i

for s in squares(5):      # 0, 1, 4, 9, 16
    print(s)
```

In short, a generator is a lightweight, lazy iterator that yields values one at a tim

# Multiple variable template

In [7]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant, Answer in {language}. Keep response under {word_limit} words."""),
    ("human", "{question}")
])

response = llm.invoke(qa_prompt.invoke(
    {
        "language": "simple English",
        "word_limit": "50",
        "question": "What is recursion?"
    }
))

print(response.content)

Recursion is when a function calls itself to solve a problem.  
It breaks a big task into smaller, similar tasks.  
Example: a function that prints numbers 5 to 1 keeps calling itself with the next number until it stops.
